# 2024 유로 스페인 빌드업 패턴: 패스 네트워크

2024 UEFA 유로에서 스페인이 치른 7경기(조별리그 3 + 토너먼트 4) 전체의 패스 네트워크를 그려 경기별로 빌드업이 어떻게 달랐는지, 대회를 관통하는 스타일은 무엇이었는지 살펴봅니다.

- 각 경기는 **첫 교체 시각 이전**의 성공한 패스만 사용해 선발 라인업이 고정된 구간만 분석합니다.
- 선수 라벨은 `lineups`의 `player_nickname`을 사용합니다 (성이 두 개인 스페인 선수 표기 오류 방지).
- 분석 기획은 [`PLAN.md`](./PLAN.md), 백로그 항목은 `ideas/backlog.md`의 "2024 유로 스페인의 빌드업 패턴"을 참고하세요.

## 방법론: 패스 네트워크를 어떻게 도출하는가

패스 네트워크는 "선수가 평균적으로 어디에 있었는지"(노드)와 "누구와 얼마나 자주 연결됐는지"(엣지)를 하나의 피치 위에 겹쳐 그린 것입니다. `plot_pass_network()`(`src/visualizer.py`)의 계산 순서는 다음과 같습니다.

1. **분석 구간을 첫 교체 시각 이전으로 제한**합니다. 선수 교체가 일어나면 그 이후 데이터는 다른 11명의 조합이 섞여 들어가 네트워크가 왜곡되므로, 선발 라인업이 고정된 구간만 사용합니다. `minute_limit`을 직접 넘기지 않으면 해당 팀의 `Substitution` 이벤트 중 가장 이른 `minute`을 자동으로 사용합니다.
2. **성공한 패스만 포함**합니다. StatsBomb 데이터는 `pass_outcome`이 `NaN`이면 성공, 문자열(`Incomplete`, `Out` 등)이면 실패를 의미합니다. 또한 `pass_recipient`가 기록된 패스만 사용합니다(코너킥처럼 특수한 경우 등 recipient가 없는 경우 제외).
3. **선수별 평균 위치(노드 좌표)**: 각 선수가 패스를 "시작한" 위치들의 평균(`x`, `y`)을 구합니다. 노드 크기는 해당 선수의 패스 시도 횟수에 비례합니다.
4. **선수 쌍 연결 강도(엣지)**: `(passer, recipient)` 쌍의 패스 횟수를 세고, 방향을 구분하지 않기 위해 `(A, B)`와 `(B, A)`를 하나의 쌍으로 합산합니다. `min_pass_count`(기본값 2) 미만으로 연결된 쌍은 노이즈로 보고 제외합니다. 엣지 굵기는 합산된 패스 횟수에 비례합니다.
5. **선수 라벨**: 기본은 이벤트 데이터의 `player`(전체 법적 이름)지만, `lineup_df`(즉 `get_match_lineups(...)[team_name]`)를 넘기면 `player_nickname`으로 표시합니다. 스페인처럼 성이 두 개인 국가의 선수는 `player` 필드를 그대로 쓰거나 마지막 단어만 잘라 쓰면 실제로 통용되는 이름과 달라지는 경우가 있어(예: `Daniel Carvajal Ramos`를 마지막 단어로 자르면 "Ramos"가 되어 다니 카르바할이 아니라 라모스로 오인됨), StatsBomb이 제공하는 `player_nickname`을 우선 사용합니다.

**한계**: 평균 위치는 이상치(예: 코너킥 세트피스 상황에서 골키퍼가 올라간 패스)에 영향을 받을 수 있고, 45분 이전만 보므로 후반전의 전술 변화는 반영되지 않습니다. 경기당 하나의 "스냅샷"이라는 점을 감안해 해석해야 합니다.

In [ ]:
import os
import sys

if sys.platform.startswith('win'):
    sys.stdout.reconfigure(encoding='utf-8')

sys.path.append(os.path.dirname(os.getcwd()))

import matplotlib.pyplot as plt
from src.data_loader import get_competition_matches, get_match_events, get_match_lineups
from src.visualizer import plot_pass_network

COMPETITION_ID = 55  # UEFA Euro
SEASON_ID = 282      # 2024
TEAM = "Spain"

output_dir = os.path.join(os.getcwd(), "processed", "spain_euro2024_pass_networks")
os.makedirs(output_dir, exist_ok=True)

In [ ]:
matches = get_competition_matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
spain_matches = matches[(matches['home_team'] == TEAM) | (matches['away_team'] == TEAM)].copy()
spain_matches = spain_matches.sort_values('match_date')
spain_matches[['match_id', 'match_date', 'home_team', 'away_team', 'home_score', 'away_score', 'competition_stage']]

In [ ]:
for _, match in spain_matches.iterrows():
    match_id = match['match_id']
    opponent = match['away_team'] if match['home_team'] == TEAM else match['home_team']
    stage = match['competition_stage']

    events = get_match_events(match_id=match_id)
    lineup = get_match_lineups(match_id=match_id)[TEAM]

    fig, ax = plot_pass_network(
        events_df=events,
        team_name=TEAM,
        lineup_df=lineup,
        title=f"Spain Pass Network - {stage} vs {opponent}",
    )

    filename = f"{stage.lower().replace(' ', '_')}_vs_{opponent.lower().replace(' ', '_')}.png"
    out_path = os.path.join(output_dir, filename)
    fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#1e1e1e')
    plt.show()
    plt.close(fig)
    print(f"저장 완료: {out_path}")

## 관찰 기록

7경기 결과를 보며 경기별 차이와 대회 전체를 관통하는 스페인 빌드업 스타일을 여기에 정리합니다.

- (예: 센터백 조합 라포르테-르 노르망이 모든 경기에서 빌드업의 축이었는지)
- (예: 상대의 압박 강도에 따라 카르바할의 평균 위치가 달라졌는지)
- (예: 결승전과 조별리그의 네트워크 밀도/연결 패턴 차이)